In [10]:
#This is the class for reading the dataset
import rasterio
import numpy as np
import torch
from torch.utils.data import Dataset
import os
import time
import glob
import matplotlib.pyplot as plt
import geopandas as gpd
from torch.utils.data import DataLoader
import gc


In [11]:
extract_dir = "South_Clear_Creek_data"
tif_files = glob.glob(os.path.join(extract_dir, "**", "*.tif"), recursive=True)
print(f"🛰️ Found {len(tif_files)} .tif file(s):")
for f in tif_files:
    print(" -", f)

print(len(tif_files))

🛰️ Found 5 .tif file(s):
 - South_Clear_Creek_data/South_Clear_Creek/Lidar_DEM_Hillshade/South_Clear_Creek_BareEarth_DEM_1m.tif
 - South_Clear_Creek_data/South_Clear_Creek/Lidar_DEM_Hillshade/South_Clear_Creek_BareEarth_Hillshade_1m.tif
 - South_Clear_Creek_data/South_Clear_Creek/Roads_Boundary/South_Clear_Creek_Roads_Mask.tif
 - South_Clear_Creek_data/South_Clear_Creek/NAIP/South_Clear_Creek_2023_NAIP_06m.tif
 - South_Clear_Creek_data/South_Clear_Creek/NAIP/South_Clear_Creek_2023_NAIP_1m.tif
5


In [12]:
def _normalize_naip(self, naip_patch):
    """
    Properly normalize NAIP data, handling very high floating point no-data values.
    
    Args:
        naip_patch: The raw NAIP data patch
        
    Returns:
        Normalized NAIP data as float32
    """
    # Convert to float32 first
    naip_float = naip_patch.astype(np.float32)
    
    # Create a mask for extremely high values that likely represent no-data
    # Often these are values like 1e38 or similar
    no_data_mask = np.logical_or(
        np.isinf(naip_float),
        np.abs(naip_float) > 1e30  # Threshold for unreasonably high values
    )
    
    # Set these no-data values to zero
    naip_float[no_data_mask] = 0.0
    
    # Normalize each band individually
    normalized = np.zeros_like(naip_float)
    
    for i in range(naip_float.shape[0]):
        band = naip_float[i]
        
        # Skip empty or invalid bands
        if np.all(band == 0) or np.isnan(band).any():
            normalized[i] = np.zeros_like(band)
            continue
        
        # Compute min and max, ignoring extreme values
        valid_mask = ~np.logical_or(no_data_mask[i], np.isnan(band))
        if np.any(valid_mask):
            valid_data = band[valid_mask]
            
            # Use percentiles to avoid outliers
            min_val = np.percentile(valid_data, 1)
            max_val = np.percentile(valid_data, 99)
            
            # Ensure we have a valid range
            if max_val - min_val < 1e-10:
                normalized[i] = np.zeros_like(band)
            else:
                # Apply normalization to [0,1] range
                normalized[i] = np.clip((band - min_val) / (max_val - min_val), 0, 1)
                
                # Ensure no-data values are set to zero
                normalized[i][no_data_mask[i]] = 0.0
        else:
            normalized[i] = np.zeros_like(band)
    
    return normalized

In [13]:
class RoadImageStreamingDataset(Dataset):
    """
    A PyTorch Dataset for streaming large geospatial raster data without loading everything into memory.
    Handles NAIP imagery, DEM data, and road mask for road detection.
    Each data source is kept separate rather than stacked as channels.
    """
    def __init__(self, naip_path, dem_path, road_mask_path, hillshade_path, ultrafast = True, watershed_bounds_path=None, boundary_buffer=0, patch_size=256, stride=128):
        """
        Initialize the streaming dataset with efficient metadata reading.
        
        Args:
            naip_path (str): Path to the NAIP imagery GeoTIFF
            dem_path (str): Path to the DEM GeoTIFF
            road_mask_path (str): Path to the road mask GeoTIFF
            hillshade_path (str): Path to the hillshade GeoTIFF
            patch_size (int): Size of the patches to extract (square)
            stride (int): Distance between consecutive patches (controls overlap)
        """
        self.naip_path = naip_path
        self.dem_path = dem_path
        self.road_mask_path = road_mask_path
        self.hillshade_path = hillshade_path  # Save the hillshade path
        self.watershed_bounds_path = watershed_bounds_path
        self.patch_size = patch_size
        self.stride = stride
        self.valid_patches_mask = None

        self.num_patches = 0
        try:
            
            
            # Get basic metadata without reading entire datasets
            with rasterio.open(naip_path) as src:
                self.height, self.width = src.height, src.width
                self.transform = src.transform
                self.crs = src.crs
                self.naip_bands = src.count
                self.naip_dtypes = src.dtypes
                self.naip_nodata = src.nodata
                
            with rasterio.open(hillshade_path) as hillshade_src:
                self.hillshade_nodata = hillshade_src.nodata
                sample_size = min(500, min(hillshade_src.height, hillshade_src.width))
                sample_window = ((0, sample_size), (0, sample_size))
                hillshade_sample = hillshade_src.read(1, window=sample_window)
            
            # Filter out nodata values if applicable
                if self.hillshade_nodata is not None:
                    hillshade_sample = hillshade_sample[hillshade_sample != self.hillshade_nodata]
            
                if len(hillshade_sample) > 0:
                    self.hillshade_mean = float(np.mean(hillshade_sample))
                    self.hillshade_std = float(np.std(hillshade_sample))
                    if self.hillshade_std < 1e-10:  # Avoid division by near-zero
                        self.hillshade_std = 1.0
                else:
                    self.hillshade_mean = 0.0
                    self.hillshade_std = 1.0
            
            # Get DEM stats from a sample (much faster)
            with rasterio.open(dem_path) as dem_src:
                self.dem_nodata = dem_src.nodata
                sample_size = min(500, min(dem_src.height, dem_src.width))
                sample_window = ((0, sample_size), (0, sample_size))
                dem_sample = dem_src.read(1, window=sample_window)
                
                # Filter out nodata values if applicable
                if self.dem_nodata is not None:
                    dem_sample = dem_sample[dem_sample != self.dem_nodata]
                
                if len(dem_sample) > 0:
                    self.dem_mean = float(np.mean(dem_sample))
                    self.dem_std = float(np.std(dem_sample))
                    if self.dem_std < 1e-10:  # Avoid division by near-zero
                        self.dem_std = 1.0
                else:
                    self.dem_mean = 0.0
                    self.dem_std = 1.0

                self.patches_h = max(1, (self.height - self.patch_size) // self.stride + 1)
                self.patches_w = max(1, (self.width - self.patch_size) // self.stride + 1)

                self.num_patches = self.patches_h * self.patches_w

            
        
        except Exception as e:
            print(f"Error during dataset initialization: {str(e)}")
            raise
        

        
        
        
        print(f"Image dimensions: {self.width}x{self.height}")
        print(f"Patch size: {patch_size}x{patch_size}, Stride: {stride}")
        print(f"NAIP bands: {self.naip_bands}")
        print(f"DEM stats - Mean: {self.dem_mean:.2f}, StdDev: {self.dem_std:.2f}")
        
        
    
    def __len__(self):
        """Return the total number of patches."""
        return self.num_patches
    
    def __getitem__(self, idx):
        """
        Get a specific patch by index, with retry logic if the NAIP patch is empty or constant.
        
        Args:
            idx (int): Index of the patch to retrieve
    
        Returns:
            dict: Dictionary containing separate tensors for each data source:
                  - 'naip': NAIP imagery tensor
                  - 'dem': DEM data tensor
                  - 'mask': Road mask tensor
                  - 'hillshade': Hillshade tensor
        """
        max_attempts = 10
        attempt = 0
        while attempt < max_attempts:
            try:
                # Compute patch index
                patch_idx = idx if attempt == 0 else np.random.randint(0, len(self))
    
                # Calculate patch coordinates
                h_idx = patch_idx // self.patches_w
                w_idx = patch_idx % self.patches_w
                h_start = h_idx * self.stride
                w_start = w_idx * self.stride
                h_end = min(h_start + self.patch_size, self.height)
                w_end = min(w_start + self.patch_size, self.width)
                if h_end - h_start < self.patch_size:
                    h_start = max(0, h_end - self.patch_size)
                if w_end - w_start < self.patch_size:
                    w_start = max(0, w_end - self.patch_size)
                window = ((h_start, h_end), (w_start, w_end))
    
                # Read raster sources
                naip_patch = self._read_raster(self.naip_path, window)
                dem_patch = self._read_raster(self.dem_path, window)
                mask_patch = self._read_raster(self.road_mask_path, window)
                hillshade_patch = self._read_raster(self.hillshade_path, window)
    
                # Normalize
                naip_normalized = self._normalize_naip(naip_patch)
                dem_normalized = self._normalize_dem(dem_patch)
                mask_normalized = (mask_patch > 0).astype(np.float32)
                hillshade_normalized = self.normalize_data(hillshade_patch, self.hillshade_nodata)
    
                # Check if NAIP patch is "blank"
                naip_min = naip_normalized.min()
                naip_max = naip_normalized.max()
                if abs(naip_max - naip_min) < 1e-5:
                    attempt += 1
                    if attempt == 1:
                        print(f"⚠️  Patch {idx} is blank (NAIP range {naip_min:.3f} to {naip_max:.3f}), retrying...")
                    continue  # Try another random patch
    
                # Convert to tensors and return
                return {
                    'naip': torch.from_numpy(naip_normalized),
                    'dem': torch.from_numpy(dem_normalized),
                    'mask': torch.from_numpy(mask_normalized),
                    'hillshade': torch.from_numpy(hillshade_normalized)
                }
    
            except Exception as e:
                print(f"❌ Error getting patch {idx} (attempt {attempt+1}): {str(e)}")
                attempt += 1
    
        # Fallback: return empty tensors
        print(f"⚠️  Returning empty patch for index {idx} after {max_attempts} attempts.")
        return {
            'naip': torch.zeros((self.naip_bands, self.patch_size, self.patch_size), dtype=torch.float32),
            'dem': torch.zeros((1, self.patch_size, self.patch_size), dtype=torch.float32),
            'mask': torch.zeros((1, self.patch_size, self.patch_size), dtype=torch.float32),
            'hillshade': torch.zeros((1, self.patch_size, self.patch_size), dtype=torch.float32),
        }



            
    def normalize_data(self, data_patch, nodata_value=None):
        """
        General normalization function for raster data, handling edge cases.
        
        Args:
            data_patch: The raw data patch to normalize
            nodata_value: Optional nodata value to handle
            
        Returns:
            Normalized data as float32, scaled to [0,1] range
        """
        # Convert to float32
        data_float = data_patch.astype(np.float32)
    
        # Handle nodata values if present
        if nodata_value is not None:
            data_float[data_float == nodata_value] = np.nan
        
        # Check for inf values
        if np.any(np.isinf(data_float)):
            data_float[np.isinf(data_float)] = np.nan
        
        valid_mask = ~np.isnan(data_float)

        # If no valid data exists, return zeros
        if not np.any(valid_mask):
            return np.zeros_like(data_float)
        
        valid_min = np.min(data_float[valid_mask])
        valid_max = np.max(data_float[valid_mask])
        
        # Handle case where all values are the same
        if np.abs(valid_max - valid_min) < 1e-10:
            # All values are effectively the same, return a uniform value (0.5)
            result = np.zeros_like(data_float)
            result[valid_mask] = 0.5
            return result
        
        # Normalize to [0,1] range
        normalized_data = np.zeros_like(data_float)
        normalized_data[valid_mask] = (data_float[valid_mask] - valid_min) / (valid_max - valid_min)
        
        # Ensure values are within [0,1] range (handle floating point precision issues)
        normalized_data = np.clip(normalized_data, 0, 1)
        
        return normalized_data
      
    def _normalize_dem(self, dem_patch):
        # Convert to float32
        dem_float = dem_patch.astype(np.float32)
    
        # Handle nodata values
        if self.dem_nodata is not None:
            dem_float[dem_float == self.dem_nodata] = np.nan
    
        # Create valid mask
        valid_mask = ~np.isnan(dem_float)
    
        # If no valid values, return all zeros
        if not np.any(valid_mask):
            return np.zeros_like(dem_float)
    
        # Compute min/max safely
        valid_min = np.nanmin(dem_float)
        valid_max = np.nanmax(dem_float)
    
        # Avoid divide-by-zero
        if abs(valid_max - valid_min) < 1e-10:
            return np.full_like(dem_float, 0.5)
    
        # Normalize to [0,1]
        normalized = (dem_float - valid_min) / (valid_max - valid_min)
        normalized = np.clip(normalized, 0, 1)
        normalized[np.isnan(normalized)] = 0  # Ensure no NaNs remain
    
        return normalized

        
    
    def _normalize_naip(self, naip_patch):
        """
        Properly normalize NAIP data, handling very high floating point no-data values
        and avoiding integer overflow during casting.
        
        Args:
            naip_patch: The raw NAIP data patch
            
        Returns:
            Normalized NAIP data as float32
        """
        # First create an empty float32 array
        normalized = np.zeros(naip_patch.shape, dtype=np.float32)
        
        # Process each band individually to avoid overflow during casting
        for i in range(naip_patch.shape[0]):
            band = naip_patch[i]
            
            # Create a mask for identifying no-data values before casting
            # This will depend on the input data type
            if band.dtype.kind == 'i' or band.dtype.kind == 'u':  # Integer types
                no_data_mask = band >= np.iinfo(band.dtype).max // 2
            else:  # Float types
                no_data_mask = np.logical_or(
                    np.isinf(band),
                    np.abs(band) > 1e30
                )
            
            # Create a safe copy with no-data values zeroed out
            safe_band = band.copy()
            safe_band[no_data_mask] = 0
            
            # Now we can safely cast to float32
            float_band = safe_band.astype(np.float32)
            
            # Skip empty bands
            if np.all(float_band == 0):
                normalized[i] = float_band
                continue
            
            # Compute percentiles on non-zero values to determine scaling
            non_zero = float_band[float_band != 0]
            if len(non_zero) > 0:
                min_val = np.percentile(non_zero, 1)
                max_val = np.percentile(non_zero, 99)
                
                # Only normalize if we have a valid range
                if max_val - min_val > 1e-10:
                    # Scale to [0,1] range
                    norm_band = np.zeros_like(float_band)
                    valid_mask = ~no_data_mask & (float_band != 0)
                    norm_band[valid_mask] = (float_band[valid_mask] - min_val) / ((max_val - min_val) + 1e-10)
                    normalized[i] = np.clip(norm_band, 0, 1)
                else:
                    normalized[i] = (float_band != 0).astype(np.float32)  # Binary mask for non-zero values
            else:
                normalized[i] = float_band
        
        return normalized
    
    def _read_raster(self, raster_path, window):
        try:
            with rasterio.open(raster_path) as src:
                patch = src.read(window=window)
    
                # Ensure patch has shape (bands, height, width)
                if patch.ndim == 2:
                    patch = patch[np.newaxis, :, :]
    
                bands = patch.shape[0]
                h_size = patch.shape[1] if patch.shape[1] else 0
                w_size = patch.shape[2] if patch.shape[2] else 0
    
                padded = np.zeros((bands, self.patch_size, self.patch_size), dtype=np.float32)
                padded[:, :h_size, :w_size] = patch[:, :h_size, :w_size]
                return padded
        except Exception as e:
            print(f"Error reading from {raster_path}: {str(e)}")
            bands = self.naip_bands if raster_path == self.naip_path else 1
            return np.zeros((bands, self.patch_size, self.patch_size), dtype=np.float32)

    
    def get_patch_coordinates(self, idx):
        """
        Get the real-world coordinates for a patch (useful for visualization).
        
        Args:
            idx (int): Patch index
            
        Returns:
            tuple: (minx, miny, maxx, maxy) coordinates in the CRS of the raster
        """
        h_idx = idx // self.patches_w
        w_idx = idx % self.patches_w
        
        h_start = h_idx * self.stride
        w_start = w_idx * self.stride
        
        h_end = min(h_start + self.patch_size, self.height)
        w_end = min(w_start + self.patch_size, self.width)
        
        # Convert pixel coordinates to world coordinates
        minx, maxy = self.transform * (w_start, h_start)
        maxx, miny = self.transform * (w_end, h_end)
        
        return (minx, miny, maxx, maxy)

    def _create_boundary_mask(self):
            """
            Create a mask of valid areas based on the watershed boundary shapefile.
            The mask will have True values for valid sampling points.
            """
            import geopandas as gpd
            from rasterio.features import rasterize
        
            try:
                # Read the watershed boundary shapefile
                watershed_boundary = gpd.read_file(self.watershed_bounds_path)
                
                # Ensure the shapefile has the same CRS as our rasters
                if watershed_boundary.crs != self.crs:
                    watershed_boundary = watershed_boundary.to_crs(self.crs)
                
                # Get shapes from the geodataframe
                shapes = [(geom, 1) for geom in watershed_boundary.geometry]
                
                # Rasterize the boundary to match our raster dimensions
                with rasterio.open(self.dem_path) as src:
                    watershed_mask = rasterize(
                        shapes,
                        out_shape=(src.height, src.width),
                        transform=src.transform,
                        fill=0,
                        default_value=1,
                        dtype=np.uint8
                    )
                
                # Create a buffer inward from the boundary if requested
                if self.boundary_buffer > 0:
                    from scipy.ndimage import binary_erosion
                    structure = np.ones((2 * self.boundary_buffer + 1, 2 * self.boundary_buffer + 1))
                    watershed_mask = binary_erosion(watershed_mask, structure=structure)
                
                # Create a mask for valid patch centers:
                # A patch center is valid if the entire patch would be inside the watershed
                half_patch = self.patch_size // 2
                valid_centers_mask = np.zeros_like(watershed_mask, dtype=bool)
                
                # For each potential patch center, check if the entire patch would be within watershed
                for h in range(half_patch, self.height - half_patch):
                    for w in range(half_patch, self.width - half_patch):
                        # Extract the region corresponding to this patch
                        patch_region = watershed_mask[
                            h - half_patch:h + half_patch,
                            w - half_patch:w + half_patch
                        ]
                        
                        # Check if all pixels in the patch are within the watershed
                        if np.all(patch_region > 0):
                            valid_centers_mask[h, w] = True
                
                self.valid_patches_mask = valid_centers_mask
                
                # Also check for inf/nan values in DEM to avoid them
                with rasterio.open(self.dem_path) as src:
                    # Sample a portion of the DEM to check for inf/nan
                    # We don't read the whole DEM to avoid memory issues
                    sample_size = 1000
                    step_h = max(1, self.height // sample_size)
                    step_w = max(1, self.width // sample_size)
                    
                    # Create a simplified invalid values mask
                    invalid_mask = np.zeros((self.height, self.width), dtype=bool)
                    
                    # Check for inf/nan in samples across the DEM
                    for h in range(0, self.height, step_h):
                        for w in range(0, self.width, step_w):
                            h_end = min(h + step_h, self.height)
                            w_end = min(w + step_w, self.width)
                            window = ((h, h_end), (w, w_end))
                            
                            # Read a chunk of the DEM
                            dem_chunk = src.read(1, window=window)
                            
                            # Check for inf/nan values
                            invalid_chunk = np.isnan(dem_chunk) | np.isinf(dem_chunk)
                                
                            # If any invalid values found in this chunk
                            if np.any(invalid_chunk):
                                invalid_mask[h:h_end, w:w_end] = True
                
                # Update valid centers to exclude patches that would include inf/nan areas
                for h in range(half_patch, self.height - half_patch):
                    for w in range(half_patch, self.width - half_patch):
                        # If this center is already invalid, skip
                        if not valid_centers_mask[h, w]:
                            continue
                        
                        # Check if any part of the patch would include invalid values
                        patch_region = invalid_mask[
                            h - half_patch:h + half_patch,
                            w - half_patch:w + half_patch
                        ]
                        
                        # If any invalid values in the patch, mark as invalid
                        if np.any(patch_region):
                            valid_centers_mask[h, w] = False
                
                # Update the valid patches mask
                self.valid_patches_mask = valid_centers_mask
                
            except Exception as e:
                print(f"Error creating boundary mask: {str(e)}")
                # If there's an error, don't filter by boundary (use all patches)
                self.valid_patches_mask = np.ones((self.height, self.width), dtype=bool)

    def _calculate_valid_patches(self):
        """
        Calculate the number of valid patches and create a mapping from
        valid patch index to (h, w) coordinates.
        """
        # Calculate all potential patches
        self.patches_h = max(1, (self.height - self.patch_size) // self.stride + 1)
        self.patches_w = max(1, (self.width - self.patch_size) // self.stride + 1)
        
        # If no watershed boundary, all patches are valid
        if self.valid_patches_mask is None:
            self.num_patches = self.patches_h * self.patches_w
            self.valid_patch_indices = None  # We'll calculate on the fly
            return
        
        # Create a list to store valid patch indices
        valid_indices = []
        
        # For each potential patch center
        for h_idx in range(self.patches_h):
            for w_idx in range(self.patches_w):
                # Calculate patch center
                h_center = h_idx * self.stride + self.patch_size // 2
                w_center = w_idx * self.stride + self.patch_size // 2
                
                # Ensure we're within bounds
                if (h_center < self.height and w_center < self.width and 
                    h_center >= 0 and w_center >= 0):
                    # Check if this center is valid
                    if self.valid_patches_mask[h_center, w_center]:
                        valid_indices.append((h_idx, w_idx))
        
        # Store the valid indices and count
        self.valid_patch_indices = valid_indices
        self.num_patches = len(valid_indices)


    def visualize_valid_patches(self, output_path=None, sample_fraction=0.01):
        """
        Create a visualization of valid patch centers to verify boundary filtering.
        
        Args:
            output_path (str, optional): Path to save the visualization
            sample_fraction (float): Fraction of valid patches to visualize (to avoid overcrowding)
        
        Returns:
            matplotlib.figure.Figure: The figure object
        """
        import matplotlib.pyplot as plt
        import numpy as np
        import random
        
        # Create figure
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Show the valid patches mask
        ax.imshow(self.valid_patches_mask, cmap='Greens', alpha=0.5)
        
        # Plot a sample of valid patch centers
        num_samples = int(len(self.valid_patch_indices) * sample_fraction)
        sample_indices = random.sample(range(len(self.valid_patch_indices)), min(num_samples, len(self.valid_patch_indices)))
        
        h_centers = []
        w_centers = []
        for idx in sample_indices:
            h_idx, w_idx = self.valid_patch_indices[idx]
            h_center = h_idx * self.stride + self.patch_size // 2
            w_center = w_idx * self.stride + self.patch_size // 2
            h_centers.append(h_center)
            w_centers.append(w_center)
        
        ax.scatter(w_centers, h_centers, color='blue', s=1, alpha=0.5)
        
        # Add some patch outlines to illustrate
        for i in range(min(10, len(sample_indices))):
            idx = sample_indices[i]
            h_idx, w_idx = self.valid_patch_indices[idx]
            h_start = h_idx * self.stride
            w_start = w_idx * self.stride
            ax.add_patch(plt.Rectangle((w_start, h_start), self.patch_size, self.patch_size, 
                                        fill=False, edgecolor='red', linewidth=1))
        
        plt.title(f"Valid Patch Centers ({self.num_patches} patches)")
        plt.xlabel("Width (pixels)")
        plt.ylabel("Height (pixels)")
        
        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"Visualization saved to {output_path}")
        
        return fig

    
    def _create_boundary_mask_ultrafast(self):
        """
        Create a mask of valid areas based on the watershed boundary shapefile.
        Uses the fastest possible approach without checking every pixel.
        """
        import geopandas as gpd
        from rasterio.features import rasterize
        
        try:
            # Read the watershed boundary shapefile
            watershed_boundary = gpd.read_file(self.watershed_bounds_path)
            
            # Ensure the shapefile has the same CRS as our rasters
            if watershed_boundary.crs != self.crs:
                watershed_boundary = watershed_boundary.to_crs(self.crs)
            
            # Create a simplified boundary with a buffer to account for patch size
            # This is the key optimization - we work with the vector data directly
            # Buffer inward by half patch size + any additional buffer
            buffer_distance = -(self.patch_size / 2 + self.boundary_buffer) * self.transform[0]
            simplified_boundary = watershed_boundary.geometry.buffer(buffer_distance)
            
            # Store the simplified boundary for fast point-in-polygon tests
            self.watershed_geometry = simplified_boundary
            
            # We'll do a sparse sampling of the DEM to create a coarse invalid value mask
            # This is much faster than checking every pixel
            with rasterio.open(self.dem_path) as src:
                # Create a very sparse grid (e.g., 100x100 points)
                grid_size = 100
                y_step = max(1, self.height // grid_size)
                x_step = max(1, self.width // grid_size)
                
                # Store coordinates of the sparse grid for invalid value checking
                self.check_coords = []
                for y in range(0, self.height, y_step):
                    for x in range(0, self.width, x_step):
                        # Convert pixel coordinates to real-world coordinates
                        lon, lat = rasterio.transform.xy(self.transform, y, x)
                        self.check_coords.append((lon, lat, y, x))
            
            # We don't create a full mask at this point - we'll check points on demand
            # This saves a massive amount of memory and computation
            
        except Exception as e:
            print(f"Error creating boundary mask: {str(e)}")
            self.watershed_geometry = None
                

    def _calculate_valid_patches_ultrafast(self):
        """
        Calculate valid patches using the fastest possible approach.
        """
        from shapely.geometry import Point
        
        # Calculate all potential patches
        self.patches_h = max(1, (self.height - self.patch_size) // self.stride + 1)
        self.patches_w = max(1, (self.width - self.patch_size) // self.stride + 1)
        
        # If no watershed boundary, all patches are valid
        if self.watershed_geometry is None:
            self.num_patches = self.patches_h * self.patches_w
            self.valid_patch_indices = None
            return
        
        # Sample a few points from the DEM to check for invalid values
        invalid_areas = set()
        if hasattr(self, 'check_coords'):
            with rasterio.open(self.dem_path) as src:
                # Check each sampling point
                for lon, lat, y, x in self.check_coords:
                    # Read a single pixel value
                    if 0 <= y < self.height and 0 <= x < self.width:
                        window = ((y, y+1), (x, x+1))
                        value = src.read(1, window=window)[0, 0]
                        
                        # If invalid, mark this region as problematic
                        if np.isnan(value) or np.isinf(value):
                            # Mark a region around this point (based on grid spacing)
                            invalid_areas.add((y // (self.height // 20), x // (self.width // 20)))
        
        # Create a list to store valid patch indices using sampling approach
        valid_indices = []
        invalid_count = 0
    
        # For a small subset of potential patches, check if they're valid
        sample_step = max(1, min(self.patches_h, self.patches_w) // 50)
        
        # First pass: check a sparse grid of patches
        for h_idx in range(0, self.patches_h, sample_step):
            for w_idx in range(0, self.patches_w, sample_step):
                # Calculate patch center in pixel coordinates
                h_center = h_idx * self.stride + self.patch_size // 2
                w_center = w_idx * self.stride + self.patch_size // 2
                
                # Convert to real-world coordinates
                lon, lat = rasterio.transform.xy(self.transform, h_center, w_center)
                point = Point(lon, lat)
            
                # Check if this center is within the simplified watershed boundary
                is_valid = point.within(self.watershed_geometry.iloc[0])
                
                # Check if this region has invalid values
                region_key = (h_center // (self.height // 20), w_center // (self.width // 20))
                has_invalid_values = region_key in invalid_areas
                
                if is_valid and not has_invalid_values:
                    valid_indices.append((h_idx, w_idx))
                else:
                    invalid_count += 1
        
        # Estimate proportion of valid patches from our sample
        sample_size = (self.patches_h // sample_step) * (self.patches_w // sample_step)
        valid_prop = len(valid_indices) / max(1, sample_size)
        
        # Extrapolate to estimate total number of valid patches
        estimated_valid = int(valid_prop * self.patches_h * self.patches_w)
    
        print(f"Estimated valid patches: {estimated_valid} ({valid_prop*100:.1f}% of total)")
        
        # For very large datasets, we can use statistical sampling to avoid checking every patch
        # This trades off perfect accuracy for massive speed improvements
        if self.patches_h * self.patches_w > 10000:  # If more than 10k potential patches
            # Clear the list and use a statistical approach
            valid_indices = []
            
            # Determine sampling density based on dataset size
            if estimated_valid > 50000:
                # For very large datasets, use an extremely sparse grid
                final_step = max(5, min(self.patches_h, self.patches_w) // 100)
            else:
                # For moderately large datasets, use a moderately sparse grid
                final_step = max(2, min(self.patches_h, self.patches_w) // 200)
            
            # Second pass: check a denser but still sparse grid
            for h_idx in range(0, self.patches_h, final_step):
                for w_idx in range(0, self.patches_w, final_step):
                    # Calculate patch center in pixel coordinates
                    h_center = h_idx * self.stride + self.patch_size // 2
                    w_center = w_idx * self.stride + self.patch_size // 2
                    
                    # Convert to real-world coordinates
                    lon, lat = rasterio.transform.xy(self.transform, h_center, w_center)
                    point = Point(lon, lat)
                    
                    # Check if this center is within the simplified watershed boundary
                    is_valid = point.within(self.watershed_geometry.iloc[0])
                    
                    # Check if this region has invalid values
                    region_key = (h_center // (self.height // 20), w_center // (self.width // 20))
                    has_invalid_values = region_key in invalid_areas
                    
                    if is_valid and not has_invalid_values:
                        valid_indices.append((h_idx, w_idx))
        
        # Store the valid indices and count
        self.valid_patch_indices = valid_indices
        self.num_patches = len(valid_indices)
        
        # Update getitem to handle sparse sampling
        self.use_sparse_sampling = True
    

In [17]:
def test_road_image_streaming_dataset(naip_path, dem_path, road_mask_path, hillshade_path):
    """
    Test the RoadImageStreamingDataset with various checks.
    
    Args:
        naip_path (str): Path to the NAIP imagery GeoTIFF
        dem_path (str): Path to the DEM GeoTIFF
        road_mask_path (str): Path to the road mask GeoTIFF
        hillshade_path (str): Path to the hillshade GeoTIFF
    """
    print("=== Testing RoadImageStreamingDataset ===")
    
    # 1. Basic initialization test
    print("\n1. Testing initialization...")
    dataset = RoadImageStreamingDataset(
        naip_path=naip_path,
        dem_path=dem_path,
        road_mask_path=road_mask_path,
        hillshade_path=hillshade_path,
        patch_size=256,
        stride=192
    )
    print(f"Dataset contains {len(dataset)} patches")
    
    # 2. Test accessing individual patches
    print("\n2. Testing patch access...")
    # Test first, middle, and last patches
    test_indices = [0, len(dataset) // 2, len(dataset) - 1]
    
    for idx in test_indices:
        print(f"\nAccessing patch {idx}:")
        
        start_time = time.time()
        patch_data = dataset[idx]
        end_time = time.time()
        
        # Check data types and shapes
        print(f"  Access time: {(end_time - start_time)*1000:.2f} ms")
        print(f"  NAIP shape: {patch_data['naip'].shape}")
        print(f"  DEM shape: {patch_data['dem'].shape}")
        print(f"  Hillshade shape: {patch_data['hillshade'].shape}")  # Added hillshade check
        print(f"  Mask shape: {patch_data['mask'].shape}")
        
        # Check value ranges
        print(f"  NAIP range: {patch_data['naip'].min().item():.3f} to {patch_data['naip'].max().item():.3f}")
        if patch_data['dem'].numel() > 0:
            print(f"  DEM range: {patch_data['dem'].min().item():.3f} to {patch_data['dem'].max().item():.3f}")
        else:
            print("  DEM range: empty patch")
            
        if patch_data['hillshade'].numel() > 0:
            print(f"  Hillshade range: {patch_data['hillshade'].min().item():.3f} to {patch_data['hillshade'].max().item():.3f}")
        else:
            print("  Hillshade range: empty patch")

        print(f"  Mask values: {torch.unique(patch_data['mask']).numpy()}")
        
        # Get real-world coordinates
        coords = dataset.get_patch_coordinates(idx)
        print(f"  Coordinates (minx, miny, maxx, maxy): {coords}")
    
    # 3. Test DataLoader compatibility
    print("\n3. Testing DataLoader compatibility...")
    loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)
    
    start_time = time.time()
    batch = next(iter(loader))
    end_time = time.time()
    
    print(f"DataLoader batch retrieval time: {(end_time - start_time)*1000:.2f} ms")
    print(f"Batch sizes:")
    print(f"  NAIP: {batch['naip'].shape}")
    print(f"  DEM: {batch['dem'].shape}")
    print(f"  Hillshade: {batch['hillshade'].shape}")  # Added hillshade batch size
    print(f"  Mask: {batch['mask'].shape}")
    
    # 4. Test memory behavior
    print("\n4. Testing memory behavior...")
    # This is an approximation since Python's memory management makes exact measurement difficult
    import psutil
    process = psutil.Process(os.getpid())
    
    mem_before = process.memory_info().rss / (1024 * 1024)
    print(f"Memory before loading patches: {mem_before:.2f} MB")
    
    # Access multiple patches sequentially
    num_test_samples = min(20, len(dataset))
    for i in range(num_test_samples):
        _ = dataset[i]
    
    mem_after = process.memory_info().rss / (1024 * 1024)
    print(f"Memory after loading {num_test_samples} patches: {mem_after:.2f} MB")
    print(f"Memory difference: {mem_after - mem_before:.2f} MB")
    
    # 5. Visualization test
    print("\n5. Testing visualization...")
    # Create a figure to visualize some samples
    fig, axes = plt.subplots(3, 4, figsize=(20, 10))  # Changed from 3x3 to 3x4 to include hillshade
    
    # Get three random indices
    random_indices = np.random.randint(0, len(dataset), size=3)
    
    for i, idx in enumerate(random_indices):
        sample = dataset[idx]
        
        # Get RGB bands from NAIP (assuming first 3 bands are RGB)
        naip = sample['naip']
        if naip.shape[0] >= 3:  # Ensure we have at least 3 bands
            rgb = naip[:3].numpy().transpose(1, 2, 0)  # Only use first 3 bands
            rgb = np.clip(rgb, 0, 1)  # Ensure in [0,1] range
        else:
            # Fallback if we don't have 3 bands
            rgb = naip[0].numpy()
            
        # Get DEM, hillshade and mask
        dem = sample['dem'][0].numpy()  # Assuming single band
        hillshade = sample['hillshade'][0].numpy()  # Assuming single band
        mask = sample['mask'][0].numpy()  # Assuming single band
        
        # Plot RGB
        axes[i, 0].imshow(rgb)
        axes[i, 0].set_title(f"NAIP RGB (Sample {idx})")
        axes[i, 0].axis('off')
        
        # Plot DEM
        dem_plot = axes[i, 1].imshow(dem, cmap='terrain')
        axes[i, 1].set_title(f"DEM (Sample {idx})")
        axes[i, 1].axis('off')
        fig.colorbar(dem_plot, ax=axes[i, 1], fraction=0.046, pad=0.04)
        
        # Plot Hillshade
        hillshade_plot = axes[i, 2].imshow(hillshade, cmap='gray')
        axes[i, 2].set_title(f"Hillshade (Sample {idx})")
        axes[i, 2].axis('off')
        fig.colorbar(hillshade_plot, ax=axes[i, 2], fraction=0.046, pad=0.04)
        
        # Plot mask
        axes[i, 3].imshow(mask, cmap='binary')
        axes[i, 3].set_title(f"Road Mask (Sample {idx})")
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig("streaming_dataset_samples.png")
    plt.close()
    print("Visualization saved to 'streaming_dataset_samples.png'")
    
    # 6. Edge case testing
    print("\n6. Testing edge cases...")
    
    # Create a dataset with different patch sizes and strides
    for patch_size in [128, 256, 512]:
        for stride in [patch_size, patch_size // 2]:
            test_dataset = RoadImageStreamingDataset(
                naip_path=naip_path,
                dem_path=dem_path,
                road_mask_path=road_mask_path,
                hillshade_path=hillshade_path,
                patch_size=patch_size,
                stride=stride
            )
            print(f"  Patch size: {patch_size}, Stride: {stride}, Patches: {len(test_dataset)}")
            
            if len(test_dataset) > 0:
                # Test the first and last patches
                first_patch = test_dataset[0]
                last_patch = test_dataset[len(test_dataset) - 1]
                
                print(f"    First patch - NAIP: {first_patch['naip'].shape}, "
                      f"DEM: {first_patch['dem'].shape}, Hillshade: {first_patch['hillshade'].shape}, "  # Added hillshade
                      f"Mask: {first_patch['mask'].shape}")
                print(f"    Last patch - NAIP: {last_patch['naip'].shape}, "
                      f"DEM: {last_patch['dem'].shape}, Hillshade: {last_patch['hillshade'].shape}, "  # Added hillshade
                      f"Mask: {last_patch['mask'].shape}")
    
    print("\n=== Test completed successfully! ===")

# Replace these with your actual file paths
naip_path = tif_files[4]
dem_path = tif_files[0]
road_mask_path = tif_files[2]
hillshade_path = tif_files[1]  # Assuming hillshade is the second file in tif_files
    
test_road_image_streaming_dataset(naip_path, dem_path, road_mask_path, hillshade_path)

=== Testing RoadImageStreamingDataset ===

1. Testing initialization...
Image dimensions: 10460x13115
Patch size: 256x256, Stride: 192
NAIP bands: 4
DEM stats - Mean: 0.00, StdDev: 1.00
Dataset contains 3618 patches

2. Testing patch access...

Accessing patch 0:
⚠️  Patch 0 is blank (NAIP range 0.000 to 0.000), retrying...
  Access time: 165.01 ms
  NAIP shape: torch.Size([4, 256, 256])
  DEM shape: torch.Size([1, 256, 256])
  Hillshade shape: torch.Size([1, 256, 256])
  Mask shape: torch.Size([1, 256, 256])
  NAIP range: 0.000 to 1.000
  DEM range: 0.000 to 1.000
  Hillshade range: 0.000 to 1.000
  Mask values: [0. 1.]
  Coordinates (minx, miny, maxx, maxy): (431955.0, 4395481.0, 432211.0, 4395737.0)

Accessing patch 1809:


/tmp/ipykernel_36137/1156270577.py:338: RuntimeWarning: overflow encountered in cast
  padded[:, :h_size, :w_size] = patch[:, :h_size, :w_size]


  Access time: 74.18 ms
  NAIP shape: torch.Size([4, 256, 256])
  DEM shape: torch.Size([1, 256, 256])
  Hillshade shape: torch.Size([1, 256, 256])
  Mask shape: torch.Size([1, 256, 256])
  NAIP range: 0.000 to 1.000
  DEM range: 0.000 to 1.000
  Hillshade range: 0.000 to 1.000
  Mask values: [0.]
  Coordinates (minx, miny, maxx, maxy): (437139.0, 4389145.0, 437395.0, 4389401.0)

Accessing patch 3617:
⚠️  Patch 3617 is blank (NAIP range 0.000 to 0.000), retrying...
  Access time: 322.07 ms
  NAIP shape: torch.Size([4, 256, 256])
  DEM shape: torch.Size([1, 256, 256])
  Hillshade shape: torch.Size([1, 256, 256])
  Mask shape: torch.Size([1, 256, 256])
  NAIP range: 0.000 to 1.000
  DEM range: 0.000 to 1.000
  Hillshade range: 0.000 to 1.000
  Mask values: [0.]
  Coordinates (minx, miny, maxx, maxy): (442131.0, 4382809.0, 442387.0, 4383065.0)

3. Testing DataLoader compatibility...
⚠️  Patch 136 is blank (NAIP range 0.000 to 0.000), retrying...
⚠️  Patch 755 is blank (NAIP range 0.000 to

In [18]:
print(tif_files[4])

South_Clear_Creek_data/South_Clear_Creek/NAIP/South_Clear_Creek_2023_NAIP_1m.tif


In [33]:
#Model
import torch.nn as nn
import torch.nn.functional as F
import time
from tqdm import tqdm
import tensorflow as tf



In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
    print(f"GPU memory growth enabled for {len(physical_devices)} device(s)")

In [ ]:
# TensorFlow wrapper for your RoadImageStreamingDataset
class TFRoadDataset:
    """
    A wrapper that converts your PyTorch streaming dataset to TensorFlow dataset
    """
    def __init__(self, road_dataset, min_road_percentage=0.0005 , check_blank=True, max_samples=None):
        """
        Initialize the TensorFlow dataset wrapper
        
        Args:
            road_dataset: The RoadImageStreamingDataset instance
            min_road_percentage: Minimum percentage of road pixels for a valid patch
            check_blank: Whether to check and skip blank/empty patches
            max_samples: Maximum number of samples to include (None for all)
        """
        self.road_dataset = road_dataset
        self.patch_size = self.road_dataset.patch_size
        self.min_road_percentage = min_road_percentage
        self.check_blank = check_blank
        self.dataset_size = len(road_dataset)
        self.max_samples = max_samples
        
        # Determine input channels from dataset
        sample_batch = None
        for idx in range(min(100, self.dataset_size)):
            try:
                sample_batch = self.road_dataset[idx]
                break
            except Exception:
                continue
        
        if sample_batch is not None:
            self.n_channels = sample_batch['naip'].shape[0] + sample_batch['dem'].shape[0] + sample_batch['hillshade'].shape[0]
        else:
            # Default to 6 channels if we can't determine (4 NAIP + 1 DEM + 1 hillshade)
            self.n_channels = 6
            
        print(f"Using {self.n_channels} input channels")
        
        # Cache known valid and invalid indices to improve efficiency
        self.known_valid_indices = set()
        self.known_invalid_indices = set()
        
        # Presample some valid indices
        self._presample_valid_indices(initial_samples=min(500, self.dataset_size))

    def generator(self):
        sample_count = 0
        max_count = self.max_samples if self.max_samples else float('inf')
    
        while sample_count < max_count:
            try:
                idx = self._find_valid_index()
                patch = self.road_dataset[idx]
                inputs, mask = self._prepare_patch(patch)
    
                # Ensure mask shape is (H, W, 1)
                if mask.ndim == 2:
                    mask = np.expand_dims(mask, axis=-1)
    
                yield inputs.astype(np.float32), mask.astype(np.float32)
                sample_count += 1
            except Exception as e:
                print(f"Error in generator: {str(e)}")
                continue


    
    def _presample_valid_indices(self, initial_samples):
        """Pre-sample some valid indices to start with"""
        print(f"Pre-sampling {initial_samples} patches to find initial valid ones...")
        
        # Sample random indices
        sample_indices = np.random.choice(
            self.dataset_size, 
            min(initial_samples, self.dataset_size), 
            replace=False
        )
        
        valid_count = 0
        for idx in tqdm(sample_indices, desc="Pre-sampling"):
            if self._check_and_add_index(idx):
                valid_count += 1
            
            # Stop if we found enough valid indices
            if valid_count >= min(100, initial_samples // 5):
                break
                
        print(f"Found {valid_count} valid patches from initial sampling")
    
    def _check_and_add_index(self, idx):
        """Check if an index is valid and add to the appropriate set"""
        # Skip if we already know about this index
        if idx in self.known_valid_indices or idx in self.known_invalid_indices:
            return idx in self.known_valid_indices
            
        try:
            # Get the patch and check road content
            patch = self.road_dataset[idx]
            
            # Check if patch is empty/blank
            if self.check_blank:
                # Convert tensors to numpy first if they're PyTorch tensors
                naip = patch['naip']
                dem = patch['dem']
                hillshade = patch['hillshade']
                mask = patch['mask']
                
                # Convert PyTorch tensors to NumPy arrays if needed
                naip_np = naip.numpy() if hasattr(naip, 'numpy') else naip
                dem_np = dem.numpy() if hasattr(dem, 'numpy') else dem
                hillshade_np = hillshade.numpy() if hasattr(hillshade, 'numpy') else hillshade
                mask_np = mask.numpy() if hasattr(mask, 'numpy') else mask
                
                # Check for NaN or zero tensors using numpy functions
                naip_valid = not (np.isnan(naip_np).any() or np.all(naip_np == 0))
                dem_valid = not (np.isnan(dem_np).any() or np.all(dem_np == 0))
                hillshade_valid = not (np.isnan(hillshade_np).any() or np.all(hillshade_np == 0))
                
                if not (naip_valid and dem_valid and hillshade_valid):
                    self.known_invalid_indices.add(idx)
                    return False
            
            # Calculate road percentage (convert to numpy if needed)
            mask = patch['mask']
            mask_np = mask.numpy() if hasattr(mask, 'numpy') else mask
            road_percentage = np.sum(mask_np) / mask_np.size
            
            # Check if road percentage meets threshold
            if road_percentage >= self.min_road_percentage:
                self.known_valid_indices.add(idx)
                return True
            else:
                self.known_invalid_indices.add(idx)
                return False
            
        except Exception as e:
            print(f"Error checking patch {idx}: {str(e)}")
            self.known_invalid_indices.add(idx)
            return False
   
    
    def _find_valid_index(self):
        """Find a valid index, either from known ones or by checking new ones"""
        if self.known_valid_indices:
            # Return a known valid index
            return np.random.choice(list(self.known_valid_indices))
        
        # Try to find a new valid index
        attempts = 0
        max_attempts = 100
        
        while attempts < max_attempts:
            idx = np.random.randint(0, self.dataset_size)
            if self._check_and_add_index(idx):
                return idx
            attempts += 1
        
        raise RuntimeError("Could not find a valid patch after multiple attempts")
    
    
    def _prepare_patch(self, patch):
        """Prepare a patch for TensorFlow training"""
        # Extract components
        naip = patch['naip']  # Shape: [C, H, W]
        dem = patch['dem']    # Shape: [1, H, W]
        mask = patch['mask']  # Shape: [C_mask, H, W]
        hillshade = patch['hillshade']  # Shape: [1, H, W]
    
        # Convert to numpy if needed
        naip = naip.numpy() if hasattr(naip, 'numpy') else naip
        dem = dem.numpy() if hasattr(dem, 'numpy') else dem
        mask = mask.numpy() if hasattr(mask, 'numpy') else mask
        hillshade = hillshade.numpy() if hasattr(hillshade, 'numpy') else hillshade
    
        # Transpose to [H, W, C]
        naip = np.transpose(naip, (1, 2, 0))
        dem = np.transpose(dem, (1, 2, 0))
        hillshade = np.transpose(hillshade, (1, 2, 0))
    
        # Fix mask if it has multiple channels
        if mask.shape[0] > 1:
            mask = mask[0:1]  # Only use the first channel
    
        mask = np.transpose(mask, (1, 2, 0))
    
        # Clean NaNs and infs
        naip = np.nan_to_num(naip, nan=0.0, posinf=1.0, neginf=0.0)
        dem = np.nan_to_num(dem, nan=0.0, posinf=1.0, neginf=0.0)
        hillshade = np.nan_to_num(hillshade, nan=0.0, posinf=1.0, neginf=0.0)
        mask = np.nan_to_num(mask, nan=0.0, posinf=1.0, neginf=0.0)
    
        # Concatenate inputs
        inputs = np.concatenate([naip, dem, hillshade], axis=-1)
    
        return inputs.astype(np.float32), mask.astype(np.float32)

    

    
    
    def create_dataset(self, batch_size=8, shuffle_buffer=100, prefetch_buffer=5):
        """
        Create a TensorFlow dataset from the streaming dataset
        """
        dataset = tf.data.Dataset.from_generator(
            self.generator,
            output_signature=(
                tf.TensorSpec(shape=(self.patch_size, self.patch_size, self.n_channels), dtype=tf.float32),
                tf.TensorSpec(shape=(self.patch_size, self.patch_size, 1), dtype=tf.float32)
            )
        )
    
        if shuffle_buffer > 0:
            dataset = dataset.shuffle(shuffle_buffer)
    
        dataset = dataset.batch(batch_size)
        dataset = dataset.prefetch(prefetch_buffer)
    
        return dataset



        

In [ ]:
def create_unet_model(input_shape, n_classes=1):
    """
    Create a simple 3-level UNet model that avoids dimension mismatches

    Args:
        input_shape: Input shape [H, W, C]
        n_classes: Number of output classes

    Returns:
        Model
    """
    inputs = tf.keras.layers.Input(input_shape)

    # Encoder
    conv1 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    conv1 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(conv1)
    pool1 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(pool1)
    conv2 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(conv2)
    pool2 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(conv2)

    # Bottleneck
    conv3 = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(pool2)
    conv3 = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(conv3)

    # Decoder - Only 2 levels
    up2 = tf.keras.layers.UpSampling2D(size=(2, 2))(conv3)
    up2 = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(up2)
    merge2 = tf.keras.layers.Concatenate()([conv2, up2])
    conv2_up = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(merge2)
    conv2_up = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(conv2_up)

    up1 = tf.keras.layers.UpSampling2D(size=(2, 2))(conv2_up)
    up1 = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(up1)
    merge1 = tf.keras.layers.Concatenate()([conv1, up1])
    conv1_up = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(merge1)
    conv1_up = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(conv1_up)

    # Output
    if n_classes == 1:
        outputs = tf.keras.layers.Conv2D(1, 1, activation='sigmoid')(conv1_up)
    else:
        outputs = tf.keras.layers.Conv2D(n_classes, 1, activation='softmax')(conv1_up)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model


In [ ]:
def create_multimodal_unet_model(input_shapes, n_classes=1):
    """
    Create a multi-input UNet model that handles dictionary inputs directly
    
    Args:
        input_shapes: Dictionary of input shapes: {'naip': (H,W,C1), 'dem': (H,W,1), 'hillshade': (H,W,1)}
        n_classes: Number of output classes
        
    Returns:
        Model that accepts dictionary inputs and outputs segmentation masks
    """
    # Create separate input layers for each data source
    naip_input = tf.keras.layers.Input(shape=input_shapes['naip'], name='naip')
    dem_input = tf.keras.layers.Input(shape=input_shapes['dem'], name='dem')
    hillshade_input = tf.keras.layers.Input(shape=input_shapes['hillshade'], name='hillshade')
    
    # NAIP imagery processing branch (main branch)
    # Encoder path with fewer levels to avoid dimension issues
    conv1_naip = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(naip_input)
    conv1_naip = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(conv1_naip)
    pool1_naip = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(conv1_naip)
    
    conv2_naip = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(pool1_naip)
    conv2_naip = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(conv2_naip)
    pool2_naip = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(conv2_naip)
    
    # DEM processing branch (simpler)
    conv1_dem = tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same')(dem_input)
    pool1_dem = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(conv1_dem)
    conv2_dem = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(pool1_dem)
    pool2_dem = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(conv2_dem)
    
    # Hillshade processing branch (simpler)
    conv1_hill = tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same')(hillshade_input)
    pool1_hill = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(conv1_hill)
    conv2_hill = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(pool1_hill)
    pool2_hill = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(conv2_hill)
    
    # Merge the features from different modalities
    merged_features = tf.keras.layers.Concatenate()([pool2_naip, pool2_dem, pool2_hill])
    
    # Bottleneck
    bottleneck = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(merged_features)
    bottleneck = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(bottleneck)
    bottleneck = tf.keras.layers.Dropout(0.5)(bottleneck)
    
    # Decoder path
    up1 = tf.keras.layers.UpSampling2D(size=(2, 2))(bottleneck)
    up1 = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(up1)
    
    # Concatenate with features from encoder (only using NAIP for skip connections to simplify)
    merge1 = tf.keras.layers.Concatenate()([conv2_naip, up1])
    conv_up1 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(merge1)
    conv_up1 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(conv_up1)
    
    up2 = tf.keras.layers.UpSampling2D(size=(2, 2))(conv_up1)
    up2 = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(up2)
    merge2 = tf.keras.layers.Concatenate()([conv1_naip, up2])
    conv_up2 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(merge2)
    conv_up2 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(conv_up2)
    
    # Output layer
    if n_classes == 1:
        outputs = tf.keras.layers.Conv2D(1, 1, activation='sigmoid', name='mask')(conv_up2)
    else:
        outputs = tf.keras.layers.Conv2D(n_classes, 1, activation='softmax', name='mask')(conv_up2)
    
    # Create the model with dictionary inputs
    model = tf.keras.Model(
        inputs={'naip': naip_input, 'dem': dem_input, 'hillshade': hillshade_input},
        outputs=outputs
    )
    
    return model

In [ ]:
# Define a custom training function with gradient accumulation
@tf.function
def train_with_accumulation(model, optimizer, loss_fn, x, y, accumulation_steps):
    """Custom training function with gradient accumulation"""
    # Initialize accumulated gradients
    accumulated_gradients = [tf.zeros_like(v) for v in model.trainable_variables]
    
    # Calculate batch size and split size
    batch_size = tf.shape(x)[0]
    split_size = batch_size // accumulation_steps
    
    # For tracking total loss
    total_loss = 0
    
    # Loop through accumulation steps
    for i in range(accumulation_steps):
        # Get mini-batch
        start_idx = i * split_size
        end_idx = (i + 1) * split_size if i < accumulation_steps - 1 else batch_size
        x_mini = x[start_idx:end_idx]
        y_mini = y[start_idx:end_idx]
        
        # Forward and backward pass
        with tf.GradientTape() as tape:
            y_pred = model(x_mini, training=True)
            loss = loss_fn(y_mini, y_pred) / tf.cast(accumulation_steps, tf.float32)
        
        # Calculate gradients
        gradients = tape.gradient(loss, model.trainable_variables)
        
        # Accumulate gradients
        accumulated_gradients = [accu_grad + grad for accu_grad, grad in zip(accumulated_gradients, gradients)]
        
        # Add to total loss
        total_loss += loss * tf.cast(accumulation_steps, tf.float32)
    
    # Apply accumulated gradients
    optimizer.apply_gradients(zip(accumulated_gradients, model.trainable_variables))
    
    return total_loss

In [ ]:
class GradientAccumulator(tf.keras.callbacks.Callback):
    def __init__(self, accumulation_steps=4):
        super().__init__()
        self.accumulation_steps = accumulation_steps
        self.gradient_accumulation = False
        self._original_train_function = None
    
    def on_train_begin(self, logs=None):
        # Store original train function
        self._original_train_function = self.model.train_function
        
        # Create a new train function with gradient accumulation
        optimizer = self.model.optimizer
        loss_fn = self.model.loss
        trainable_variables = self.model.trainable_variables
        
        @tf.function
        def train_function(data):
            # Get the data
            x, y = data
            # Split batch into smaller chunks
            batch_size = tf.shape(x)[0]
            split_size = batch_size // self.accumulation_steps
            
            # Initialize accumulated gradients
            accumulated_gradients = [tf.zeros_like(v) for v in trainable_variables]
            
            # Total loss and metrics for tracking
            total_loss = 0
            
            # Process batch in chunks
            for i in range(self.accumulation_steps):
                start_idx = i * split_size
                end_idx = tf.minimum((i + 1) * split_size, batch_size)
                
                x_mini = x[start_idx:end_idx]
                y_mini = y[start_idx:end_idx]
                
                with tf.GradientTape() as tape:
                    y_pred = self.model(x_mini, training=True)
                    loss = loss_fn(y_mini, y_pred) / tf.cast(self.accumulation_steps, tf.float32)
                
                gradients = tape.gradient(loss, trainable_variables)
                accumulated_gradients = [(acc_g + g) for acc_g, g in zip(accumulated_gradients, gradients)]
                total_loss += loss * tf.cast(self.accumulation_steps, tf.float32)
            
            # Apply accumulated gradients
            optimizer.apply_gradients(zip(accumulated_gradients, trainable_variables))
            
            # Update metrics
            self.model.compiled_metrics.update_state(y, self.model(x, training=False))
            
            # Return metrics
            return_metrics = {'loss': total_loss}
            for metric in self.model.metrics:
                return_metrics[metric.name] = metric.result()
            
            return return_metrics
        
        # Replace the model's train function
        self.model.train_function = train_function
    
    def on_train_end(self, logs=None):
        # Restore original train function
        if self._original_train_function is not None:
            self.model.train_function = self._original_train_function

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1.0):
    """
    Calculate Dice coefficient for evaluating segmentation quality
    
    Args:
        y_true: Ground truth masks
        y_pred: Predicted masks
        smooth: Smoothing factor to avoid division by zero
        
    Returns:
        Dice coefficient
    """
    # Cast to same dtype to avoid mixed precision errors
    y_true = tf.cast(y_true, y_pred.dtype)
    
    # Flatten the predictions and targets
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    
    # Calculate intersection and sum
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    sum_true = tf.reduce_sum(y_true_f)
    sum_pred = tf.reduce_sum(y_pred_f)
    
    # Calculate Dice coefficient
    dice = (2. * intersection + smooth) / (sum_true + sum_pred + smooth)
    return dice

In [ ]:
def combined_loss(y_true, y_pred, road_weight=8.0):
    y_true = tf.cast(y_true, y_pred.dtype)
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
    weights = y_true * (road_weight - 1.0) + 1.0
    weighted_bce = tf.reduce_mean(weights * bce)
    dice = dice_coefficient(y_true, y_pred, smooth=1e-5)
    dice_loss = 1.0 - dice
    return 0.3 * weighted_bce + 0.7 * dice_loss

In [ ]:
# Custom callback for visualization during training
class VisualizationCallback(tf.keras.callbacks.Callback):
    """
    Callback to visualize predictions during training
    """
    def __init__(self, dataset, output_dir, num_samples=4, interval=5):
        """
        Initialize visualization callback
        
        Args:
            dataset: TFRoadDataset instance
            output_dir: Directory to save visualizations
            num_samples: Number of samples to visualize
            interval: Epoch interval for visualization
        """
        super().__init__()
        self.dataset = dataset
        self.output_dir = os.path.join(output_dir, 'visualizations')
        self.num_samples = num_samples
        self.interval = interval
        
        # Create output directory
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Get sample patches for visualization
        self.sample_patches = []
        for _ in range(self.num_samples):
            try:
                idx = self.dataset._find_valid_index()
                patch = self.dataset.road_dataset[idx]
                inputs, mask = self.dataset._prepare_patch(patch)
                self.sample_patches.append((idx, inputs, mask))
            except Exception as e:
                print(f"Error getting sample patch: {str(e)}")
    
    def on_epoch_end(self, epoch, logs=None):
        # Only visualize at specified intervals
        if (epoch + 1) % self.interval != 0:
            return
        
        for i, (idx, inputs, mask) in enumerate(self.sample_patches):
            try:
                # Get prediction
                pred = self.model.predict(np.expand_dims(inputs, axis=0), verbose=0)
                pred = np.squeeze(pred, axis=0)
                
                # Create visualization
                fig = self._visualize_prediction(inputs, mask, pred)
                
                # Save visualization
                plt.savefig(os.path.join(self.output_dir, f'epoch_{epoch+1}_sample_{i+1}.png'))
                plt.close(fig)
            except Exception as e:
                print(f"Error visualizing sample {i}: {str(e)}")
    
    def _visualize_prediction(self, inputs, mask, pred, threshold=0.5):
        """Create visualization of input, ground truth, and prediction"""
        # Extract RGB channels for visualization (assuming first 3 channels are RGB)
        if inputs.shape[-1] >= 3:
            rgb = inputs[..., :3]
        else:
            # If not enough channels, use grayscale
            rgb = np.repeat(inputs[..., 0:1], 3, axis=-1)
        
        # Clip values to valid range
        rgb = np.clip(rgb, 0, 1)
        
        # Get components
        dem = inputs[..., -2] if inputs.shape[-1] >= 5 else None
        hillshade = inputs[..., -1] if inputs.shape[-1] >= 6 else None
        
        # Threshold prediction
        pred_binary = (pred > threshold).astype(np.float32)
        
        # Create figure
        if dem is not None and hillshade is not None:
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            
            # RGB image
            axes[0, 0].imshow(rgb)
            axes[0, 0].set_title('RGB Image')
            axes[0, 0].axis('off')
            
            # DEM
            dem_vis = axes[0, 1].imshow(dem, cmap='terrain')
            axes[0, 1].set_title('DEM')
            axes[0, 1].axis('off')
            plt.colorbar(dem_vis, ax=axes[0, 1], fraction=0.046, pad=0.04)
            
            # Hillshade
            axes[0, 2].imshow(hillshade, cmap='gray')
            axes[0, 2].set_title('Hillshade')
            axes[0, 2].axis('off')
            
            # Ground truth
            axes[1, 0].imshow(rgb)
            axes[1, 0].imshow(np.squeeze(mask), alpha=0.5, cmap='Reds')
            axes[1, 0].set_title('Ground Truth')
            axes[1, 0].axis('off')
            
            # Prediction probability
            axes[1, 1].imshow(rgb)
            pred_vis = axes[1, 1].imshow(np.squeeze(pred), alpha=0.7, cmap='plasma')
            axes[1, 1].set_title('Prediction (Probability)')
            axes[1, 1].axis('off')
            plt.colorbar(pred_vis, ax=axes[1, 1], fraction=0.046, pad=0.04)
            
            # Binary prediction
            axes[1, 2].imshow(rgb)
            axes[1, 2].imshow(np.squeeze(pred_binary), alpha=0.5, cmap='Reds')
            axes[1, 2].set_title(f'Binary Prediction (t={threshold})')
            axes[1, 2].axis('off')
        else:
            # Simplified visualization if DEM/hillshade not available
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            
            # RGB image with ground truth
            axes[0].imshow(rgb)
            axes[0].imshow(np.squeeze(mask), alpha=0.5, cmap='Reds')
            axes[0].set_title('Ground Truth')
            axes[0].axis('off')
            
            # RGB image with prediction probability
            axes[1].imshow(rgb)
            pred_vis = axes[1].imshow(np.squeeze(pred), alpha=0.7, cmap='plasma')
            axes[1].set_title('Prediction (Probability)')
            axes[1].axis('off')
            
            # RGB image with binary prediction
            axes[2].imshow(rgb)
            axes[2].imshow(np.squeeze(pred_binary), alpha=0.5, cmap='Reds')
            axes[2].set_title(f'Binary Prediction (t={threshold})')
            axes[2].axis('off')
        
        plt.tight_layout()
        return fig

In [ ]:
def train_road_detector(road_dataset, output_dir, batch_size=2, epochs=50, 
                       min_road_percentage=0.0005, check_blank=True,
                       train_samples=1000, val_samples=200, 
                       lr=1e-4, visualization_interval=5,
                       accumulation_steps=4):
    """
    Train the road detection model with TensorFlow
    
    Args:
        road_dataset: The RoadImageStreamingDataset instance
        output_dir: Directory to save model checkpoints and visualizations
        batch_size: Batch size for training
        epochs: Number of epochs to train
        min_road_percentage: Minimum percentage of road pixels for a valid patch
        check_blank: Whether to check and skip blank/empty patches
        train_samples: Number of samples to use for training per epoch
        val_samples: Number of samples to use for validation per epoch
        lr: Learning rate
        visualization_interval: Interval (in epochs) for visualization
        accumulation_steps: Number of gradient accumulation steps
    """
    import os
    os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
    os.environ['XLA_FLAGS'] = '--xla_gpu_strict_conv_algorithm_picker=false'
    mixed_precision = True
    # Disable XLA JIT compilation
    tf.config.optimizer.set_jit(False)
    
    # Enable memory growth to avoid allocating all GPU memory at once
    gpus = tf.config.experimental.list_physical_devices('GPU')
    
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            
    if mixed_precision:
        policy = tf.keras.mixed_precision.Policy('mixed_float16')
        tf.keras.mixed_precision.set_global_policy(policy)
        
    print("Using mixed precision training")
    print(f"Using reduced batch size {batch_size} instead of gradient accumulation")
    os.makedirs(output_dir, exist_ok=True)
    
    # Create TensorFlow datasets
    print("Creating training dataset...")
    train_tf_dataset = TFRoadDataset(
        road_dataset=road_dataset,
        min_road_percentage=min_road_percentage,
        check_blank=check_blank,
        max_samples=train_samples
    )
    
    print("Creating validation dataset...")
    val_tf_dataset = TFRoadDataset(
        road_dataset=road_dataset,
        min_road_percentage=min_road_percentage,
        check_blank=check_blank,
        max_samples=val_samples
    )
    
    # Get TensorFlow data loaders
    train_dataset = train_tf_dataset.create_dataset(
        batch_size=batch_size,
        shuffle_buffer=min(200, train_samples // 2),
        prefetch_buffer=tf.data.AUTOTUNE
    )
    
    val_dataset = val_tf_dataset.create_dataset(
        batch_size=batch_size,
        shuffle_buffer=min(100, val_samples // 2),
        prefetch_buffer=tf.data.AUTOTUNE
    )
    
    # Get input shape from a sample
    for batch in train_dataset.take(1):
        inputs, _ = batch  # Correctly unpack the tuple
        input_shape = inputs.shape[1:]
        break
    
    # Create model
    model = create_unet_model(input_shape=input_shape)
    
    def combined_loss_fn(road_weight=8.0):
        def loss(y_true, y_pred):
            return combined_loss(y_true, y_pred, road_weight=road_weight)
        return loss


    
    # Compile model with standard optimizer
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    model.compile(
        optimizer=optimizer,
        loss=combined_loss_fn(),  # ✅ Now you're passing the actual loss function
        metrics=[dice_coefficient, 'binary_accuracy']
    )



    
    # Print model summary
    model.summary()
    
    # Callbacks
    callbacks = [
        # Model checkpoint to save best model
        tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(output_dir, 'best_model.h5'),
            monitor='val_dice_coefficient',
            mode='max',
            save_best_only=True,
            verbose=1
        ),
        # Learning rate reduction on plateau
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            verbose=1,
            min_lr=1e-6
        ),
        # Early stopping if model stops improving
        tf.keras.callbacks.EarlyStopping(
            monitor='val_dice_coefficient',
            mode='max',
            patience=15,
            verbose=1,
            restore_best_weights=True
        ),
        # TensorBoard logging
        tf.keras.callbacks.TensorBoard(
            log_dir=os.path.join(output_dir, 'logs'),
            histogram_freq=1
        )
    ]
    
    # Add memory cleanup callback
    class MemoryCleanupCallback(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            import gc
            gc.collect()
            tf.keras.backend.clear_session()
    
    callbacks.append(MemoryCleanupCallback())
    
    # Train model
    print(f"Starting training for {epochs} epochs...")
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=callbacks
    )
    
    # Save final model
    model.save(os.path.join(output_dir, 'final_model.h5'))
    
    # Plot training history
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss')
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['dice_coefficient'], label='Train Dice')
    plt.plot(history.history['val_dice_coefficient'], label='Val Dice')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Coefficient')
    plt.legend()
    plt.title('Training and Validation Dice Coefficient')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_history.png'))
    plt.close()
    
    print(f"Training completed. Results saved to {output_dir}")
    return model

In [ ]:
# Inference function for prediction
def predict_roads(model, road_dataset, idx, threshold=0.5):
    """
    Predict roads for a single patch
    
    Args:
        model: Trained TensorFlow model
        road_dataset: RoadImageStreamingDataset instance
        idx: Index of patch to predict
        threshold: Threshold for binary prediction
        
    Returns:
        Dictionary with prediction results and visualization
    """
    # Get patch
    patch = road_dataset[idx]
    
    # Prepare patch for TensorFlow
    tf_dataset = TFRoadDataset(road_dataset)
    inputs, gt_mask = tf_dataset._prepare_patch(patch)
    
    # Add batch dimension
    inputs_batch = np.expand_dims(inputs, axis=0)
    
    # Run prediction
    pred = model.predict(inputs_batch, verbose=0)
    pred = np.squeeze(pred)
    
    # Threshold to get binary mask
    pred_binary = (pred > threshold).astype(np.float32)
    
    # Extract RGB for visualization
    if inputs.shape[-1] >= 3:
        rgb = inputs[..., :3]
    else:
        rgb = np.repeat(inputs[..., 0:1], 3, axis=-1)
    
    rgb = np.clip(rgb, 0, 1)
    
    # Calculate metrics
    gt_mask_flat = gt_mask.flatten()
    pred_flat = pred_binary.flatten()
    
    # Calculate Dice coefficient
    intersection = np.sum(gt_mask_flat * pred_flat)
    dice = (2. * intersection) / (np.sum(gt_mask_flat) + np.sum(pred_flat) + 1e-6)
    
    # Calculate IoU
    union = np.sum(gt_mask_flat) + np.sum(pred_flat) - intersection
    iou = intersection / (union + 1e-6)
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Ground truth
    axes[0].imshow(rgb)
    axes[0].imshow(np.squeeze(gt_mask), alpha=0.5, cmap='Reds')
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')
    
    # Prediction probability
    axes[1].imshow(rgb)
    pred_vis = axes[1].imshow(pred, alpha=0.7, cmap='plasma')
    axes[1].set_title('Prediction (Probability)')
    axes[1].axis('off')
    plt.colorbar(pred_vis, ax=axes[1], fraction=0.046, pad=0.04)
    
    # Binary prediction
    axes[2].imshow(rgb)
    axes[2].imshow(pred_binary, alpha=0.5, cmap='Reds')
    axes[2].set_title(f'Binary Prediction (Dice={dice:.4f}, IoU={iou:.4f})')
    axes[2].axis('off')
    
    plt.tight_layout()
    
    return {
        'prediction': pred,
        'binary': pred_binary,
        'dice': dice,
        'iou': iou,
        'visualization': fig
    }

In [ ]:
# Evaluation function for model assessment
def evaluate_model(model, road_dataset, output_dir, num_samples=20, threshold=0.5,
                  min_road_percentage=0.01, check_blank=True):
    """
    Evaluate the model on multiple patches and save results
    
    Args:
        model: Trained TensorFlow model
        road_dataset: RoadImageStreamingDataset instance
        output_dir: Directory to save evaluation results
        num_samples: Number of samples to evaluate
        threshold: Threshold for binary prediction
        min_road_percentage: Minimum percentage of road pixels for valid samples
        check_blank: Whether to check and skip blank/empty patches
        
    Returns:
        Dictionary with evaluation metrics
    """
    eval_dir = os.path.join(output_dir, 'evaluation')
    os.makedirs(eval_dir, exist_ok=True)
    
    print(f"Evaluating model on {num_samples} samples...")
    
    # Create dataset wrapper to find valid patches
    tf_dataset = TFRoadDataset(
        road_dataset=road_dataset,
        min_road_percentage=min_road_percentage,
        check_blank=check_blank
    )
    
    # Get valid indices
    valid_indices = []
    attempts = 0
    max_attempts = num_samples * 5
    
    while len(valid_indices) < num_samples and attempts < max_attempts:
        idx = tf_dataset._find_valid_index()
        if idx not in valid_indices:
            valid_indices.append(idx)
        attempts += 1
    
    print(f"Found {len(valid_indices)} valid patches for evaluation")
    
    # Calculate metrics
    dice_scores = []
    iou_scores = []
    
    for i, idx in enumerate(tqdm(valid_indices, desc="Evaluating")):
        try:
            # Predict roads for this patch
            result = predict_roads(model, road_dataset, idx, threshold)
            
            # Save visualization
            plt.savefig(os.path.join(eval_dir, f'sample_{i+1}_dice_{result["dice"]:.4f}.png'))
            plt.close(result['visualization'])
            
            # Save metrics
            dice_scores.append(result['dice'])
            iou_scores.append(result['iou'])
            
        except Exception as e:
            print(f"Error evaluating patch {idx}: {str(e)}")
            continue
    
    # Calculate summary statistics
    if len(dice_scores) > 0:
        mean_dice = np.mean(dice_scores)
        std_dice = np.std(dice_scores)
        mean_iou = np.mean(iou_scores)
        std_iou = np.std(iou_scores)
        
        # Create summary visualization
        plt.figure(figsize=(10, 6))
        
        plt.subplot(1, 2, 1)
        plt.hist(dice_scores, bins=10, color='skyblue', edgecolor='black')
        plt.axvline(mean_dice, color='red', linestyle='--', linewidth=2)
        plt.text(mean_dice, plt.ylim()[1]*0.9, f' Mean: {mean_dice:.4f}', color='red')
        plt.xlabel('Dice Coefficient')
        plt.ylabel('Frequency')
        plt.title('Distribution of Dice Scores')
        
        plt.subplot(1, 2, 2)
        plt.hist(iou_scores, bins=10, color='lightgreen', edgecolor='black')
        plt.axvline(mean_iou, color='red', linestyle='--', linewidth=2)
        plt.text(mean_iou, plt.ylim()[1]*0.9, f' Mean: {mean_iou:.4f}', color='red')
        plt.xlabel('IoU')
        plt.ylabel('Frequency')
        plt.title('Distribution of IoU Scores')
        
        plt.tight_layout()
        plt.savefig(os.path.join(eval_dir, 'metrics_summary.png'))
        plt.close()
        
        # Save metrics to file
        metrics = {
            'mean_dice': float(mean_dice),
            'std_dice': float(std_dice),
            'mean_iou': float(mean_iou),
            'std_iou': float(std_iou),
            'num_samples': len(dice_scores)
        }
        
        # Save as JSON
        import json
        with open(os.path.join(eval_dir, 'metrics.json'), 'w') as f:
            json.dump(metrics, f, indent=4)
        
        # Print summary
        print(f"Evaluation completed on {len(dice_scores)} samples:")
        print(f"Mean Dice Score: {mean_dice:.4f} ± {std_dice:.4f}")
        print(f"Mean IoU: {mean_iou:.4f} ± {std_iou:.4f}")
        
        return metrics
    else:
        print("No valid samples were successfully evaluated.")
        return {
            'mean_dice': 0.0,
            'std_dice': 0.0,
            'mean_iou': 0.0,
            'std_iou': 0.0,
            'num_samples': 0
        }

In [ ]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')


naip_path = tif_files[3]
dem_path = tif_files[0]
road_mask_path = tif_files[4]
hillshade_path = tif_files[1] 
output_dir = "tf_road_detection_results"

# Use the RoadImageStreamingDataset class that's defined in your notebook
# This assumes the class is already defined in the current scope

# Create dataset (without loading everything into memory)
dataset = RoadImageStreamingDataset(
    naip_path=naip_path,
    dem_path=dem_path,
    road_mask_path=road_mask_path,
    hillshade_path=hillshade_path,
    patch_size=160,
    stride=128
)
    
# Train model using streaming approach (with the example_usage flag set to True)
model = train_road_detector(
    road_dataset=dataset,
    output_dir=output_dir,
    batch_size=1,
    epochs=50,                 # Will only run 1 epoch in example mode
    min_road_percentage=0.0005,  # At least 1% road content
    check_blank=True,          # Skip blank/invalid patches
    train_samples=1000,        # Use 1000 valid patches per epoch for training
    val_samples=200,           # Use 200 valid patches for validation
    lr=1e-4,
    visualization_interval=1,
    accumulation_steps=4 # Visualize every epoch in example mode
         # Set to False for full training
)

# Evaluate model on a few samples
evaluate_model(
    model=model,
    road_dataset=dataset,
    output_dir=output_dir,
    num_samples=5,             # Only evaluate 5 samples in example mode
    threshold=0.5,
    min_road_percentage=0.01,
    check_blank=True
)

In [ ]:
nvidia-smi

In [ ]:
def visualize_road_predictions(model, dataset, num_samples=4, output_path=None):
    """
    Visualize road detection predictions
    
    Args:
        model: Trained TensorFlow model
        dataset: TFRoadDataset instance or TensorFlow dataset
        num_samples: Number of samples to visualize
        output_path: Optional path to save visualization
    """
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Check if dataset is TFRoadDataset or TensorFlow dataset
    if hasattr(dataset, 'create_dataset'):
        # It's a TFRoadDataset, create a TensorFlow dataset
        tf_dataset = dataset.create_dataset(batch_size=1)
    else:
        # It's already a TensorFlow dataset
        tf_dataset = dataset
    
    # Create figure
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    
    # Get samples
    samples = []
    for batch in tf_dataset.take(num_samples):
        samples.append(batch)
    
    # Process each sample
    for i, (inputs, ground_truth) in enumerate(samples):
        # Get prediction
        prediction = model.predict(inputs)
        
        # Convert to numpy arrays and ensure they're in the right shape
        if isinstance(inputs, tf.Tensor):
            inputs = inputs.numpy()
        if isinstance(ground_truth, tf.Tensor):
            ground_truth = ground_truth.numpy()
        if isinstance(prediction, tf.Tensor):
            prediction = prediction.numpy()
        
        # Handle batch dimension
        if inputs.shape[0] == 1:
            inputs = inputs[0]
        if ground_truth.shape[0] == 1:
            ground_truth = ground_truth[0]
        if prediction.shape[0] == 1:
            prediction = prediction[0]
        
        # Display input image (first 3 channels for RGB visualization)
        display_img = inputs[:, :, :3] if inputs.shape[-1] > 3 else inputs
        
        # Ensure values are in [0, 1] range for display
        if display_img.max() > 1.0:
            display_img = display_img / 255.0
        
        axes[i, 0].imshow(display_img)
        axes[i, 0].set_title('Input Image')
        axes[i, 0].axis('off')
        
        # Display ground truth mask
        gt_display = ground_truth[:, :, 0] if ground_truth.ndim > 2 else ground_truth
        axes[i, 1].imshow(gt_display, cmap='Greens', vmin=0, vmax=1)
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Display prediction
        pred_display = prediction[:, :, 0] if prediction.ndim > 2 else prediction
        axes[i, 2].imshow(pred_display, cmap='Greens', vmin=0, vmax=1)
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')
    
    # Adjust layout
    plt.tight_layout()
    
    # Save if output path is provided
    if output_path:
        plt.savefig(output_path, dpi=200, bbox_inches='tight')
        print(f"Visualization saved to {output_path}")
    
    # Show plot
    plt.show()
    
    return fig

In [ ]:
visualize_road_predictions(
    model=model,
    dataset=val_tf_dataset,  # Your TFRoadDataset instance
    num_samples=4,
    output_path="road_predictions.png"
)

In [ ]:
# Create a dataset for visualization
val_tf_dataset = TFRoadDataset(
    road_dataset=dataset,  # Your main dataset
    min_road_percentage=0.01,
    check_blank=True,
    max_samples=10  # Small number for visualization
)

# Create the TensorFlow dataset
val_dataset = val_tf_dataset.create_dataset(batch_size=1)

# Visualize the results
fig = visualize_model_predictions(
    model=model,
    dataset=val_dataset,
    num_samples=4,
    output_path="road_detection_results.png"
)

In [ ]:
!nvidia-smi
